## 1. Load Projection and Entity Data

Load the aligned projection outputs plus upstream tables that help explain where matching and normalization are failing.

In [ ]:
from __future__ import annotations

from difflib import SequenceMatcher
import re
import sys
import unicodedata
from pathlib import Path

import pandas as pd


def find_repo_root(start_path: Path) -> Path:
    for parent in [start_path] + list(start_path.parents):
        if (parent / "speakermining" / "src" / "process").exists() and (parent / "data").exists():
            return parent
    raise RuntimeError(f"Could not find repository root from {start_path}")


repo_root = find_repo_root(Path.cwd())
sys.path.insert(0, str(repo_root / "speakermining" / "src"))

from process.entity_disambiguation.config import CORE_CLASSES, get_aligned_csv_path


def read_csv(path: Path) -> pd.DataFrame:
    return pd.read_csv(path) if path.exists() else pd.DataFrame()


def strip_accents(value: object) -> str:
    text = "" if pd.isna(value) else str(value)
    normalized = unicodedata.normalize("NFKD", text)
    return "".join(ch for ch in normalized if not unicodedata.combining(ch))


def strip_episode_date_tokens(value: object) -> str:
    text = "" if pd.isna(value) else str(value)
    text = re.sub(r"\b\d{1,2}[.\-/]\d{1,2}[.\-/]\d{2,4}\b", " ", text)
    text = re.sub(r"\b\d{1,2}\s+\d{1,2}\s+\d{2,4}\b", " ", text)
    return text


def normalize_text(value: object) -> str:
    text = strip_accents(value).casefold()
    text = re.sub(r"[\u2010-\u2015\-_/|]+", " ", text)
    text = re.sub(r"[^\w\s]+", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def normalize_name(value: object) -> str:
    text = normalize_text(value)
    text = re.sub(r"\b(der|die|das|von|vom|und|the|dr|prof|mr|mrs|ms)\b", " ", text)
    return re.sub(r"\s+", " ", text).strip()


def normalize_name_relaxed(value: object) -> str:
    text = normalize_name(value)
    text = re.sub(r"\([^)]*\)", " ", text)
    text = re.sub(r"\b(alias|genannt|geb\.)\b", " ", text)
    return re.sub(r"\s+", " ", text).strip()


def normalize_episode_title(value: object) -> str:
    text = strip_episode_date_tokens(value)
    text = strip_accents(text).casefold()
    text = re.sub(r"[\u2010-\u2015\-_/|]+", " ", text)
    text = re.sub(r"[^\w\s]+", " ", text)
    text = re.sub(r"\b(sendung vom|folge|staffel|episode|episoden|vom|von)\b", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def normalize_episode_title_relaxed(value: object) -> str:
    text = normalize_episode_title(value)
    text = re.sub(r"\([^)]*\)", " ", text)
    text = re.sub(r"\b(?:vom|von)\b", " ", text)
    return re.sub(r"\s+", " ", text).strip()


def normalize_date_string(value: object) -> str:
    if pd.isna(value):
        return ""
    parsed = pd.to_datetime(str(value).strip(), dayfirst=True, errors="coerce")
    if pd.isna(parsed):
        return ""
    return parsed.strftime("%Y-%m-%d")


def normalize_time_string(value: object) -> str:
    if pd.isna(value):
        return ""
    text = str(value).strip()
    return text if re.fullmatch(r"\d{2}:\d{2}:\d{2}", text) else ""


def similarity(left: object, right: object) -> float:
    return SequenceMatcher(None, normalize_text(left), normalize_text(right)).ratio()


def best_match_index(source_value: object, candidates: pd.Series, normalizer=normalize_text) -> tuple[int | None, float, object]:
    source_norm = normalizer(source_value)
    best_index = None
    best_score = -1.0
    best_value = None
    for idx, candidate_value in candidates.items():
        candidate_norm = normalizer(candidate_value)
        if not source_norm and not candidate_norm:
            score = 1.0
        else:
            score = SequenceMatcher(None, source_norm, candidate_norm).ratio()
        if score > best_score:
            best_index = idx
            best_score = score
            best_value = candidate_value
    return best_index, best_score, best_value


projection_dir = repo_root / "data" / "31_entity_disambiguation" / "projections"
candidate_dir = repo_root / "data" / "20_candidate_generation"
mention_dir = repo_root / "data" / "10_mention_detection"

projection_paths = {core_class: get_aligned_csv_path(core_class) for core_class in CORE_CLASSES}
projections = {core_class: read_csv(path) for core_class, path in projection_paths.items()}
episodes_df = projections.get("episodes", pd.DataFrame())
persons_df = projections.get("persons", pd.DataFrame())
broadcasting_programs_df = projections.get("broadcasting_programs", pd.DataFrame())
series_df = projections.get("series", pd.DataFrame())
topics_df = projections.get("topics", pd.DataFrame())
roles_df = projections.get("roles", pd.DataFrame())
organizations_df = projections.get("organizations", pd.DataFrame())

candidate_episodes = read_csv(candidate_dir / "episodes.csv")
mention_persons = read_csv(mention_dir / "persons.csv")
mention_episodes = read_csv(mention_dir / "episodes.csv")
candidate_persons = mention_persons.copy()

print(f"repo_root={repo_root}")
print(f"projection_tables={sorted(projections)}")
print(f"episodes_rows={len(episodes_df)} persons_rows={len(persons_df)}")
print(f"candidate_episodes_rows={len(candidate_episodes)} mention_persons_rows={len(mention_persons)} mention_episodes_rows={len(mention_episodes)}")

## 2. Inspect Projection Outputs Step by Step

Summarize the aligned outputs and highlight where rows are still unresolved or missing source context.

In [ ]:
expected_columns = {
    "episodes": [
        "episode_key",
        "broadcasting_program_key",
        "source_zdf_value",
        "source_wikidata_value",
        "deterministic_alignment_method",
        "deterministic_alignment_status",
    ],
    "persons": [
        "person_key",
        "source_zdf_value",
        "source_fernsehserien_value",
        "deterministic_alignment_method",
        "deterministic_alignment_status",
    ],
}

inventory_rows = []
for core_class, frame in projections.items():
    status_column = next((column for column in ["deterministic_alignment_status", "alignment_status", "status"] if column in frame.columns), None)
    source_columns = [column for column in frame.columns if column.startswith("source_")]
    unresolved_rows = 0
    if status_column:
        unresolved_rows = int(frame[status_column].astype(str).str.casefold().eq("unresolved").sum())
    inventory_rows.append(
        {
            "core_class": core_class,
            "rows": len(frame),
            "columns": len(frame.columns),
            "unresolved_rows": unresolved_rows,
            "source_columns": len(source_columns),
            "missing_expected_columns": ", ".join(column for column in expected_columns.get(core_class, []) if column not in frame.columns),
        }
    )

inventory_df = pd.DataFrame(inventory_rows).sort_values(["unresolved_rows", "rows", "core_class"], ascending=[False, False, True])
print(inventory_df.to_string(index=False))

for core_class in ["episodes", "persons"]:
    frame = projections.get(core_class, pd.DataFrame())
    if frame.empty:
        continue
    print()
    print(f"[{core_class}] columns={list(frame.columns)}")
    status_column = next((column for column in ["deterministic_alignment_status", "alignment_status", "status"] if column in frame.columns), None)
    if status_column:
        print(frame[status_column].astype(str).value_counts(dropna=False).to_string())

## 3. Normalize Raw Labels Before Comparing

In [ ]:
def add_episode_normalizations(frame: pd.DataFrame, title_column: str, date_column: str | None = None, time_column: str | None = None) -> pd.DataFrame:
    result = frame.copy()
    if title_column in result.columns:
        result["title_current"] = result[title_column].map(normalize_episode_title)
        result["title_relaxed"] = result[title_column].map(normalize_episode_title_relaxed)
    else:
        result["title_current"] = ""
        result["title_relaxed"] = ""
    if date_column and date_column in result.columns:
        result["date_norm"] = result[date_column].map(normalize_date_string)
    else:
        result["date_norm"] = ""
    if time_column and time_column in result.columns:
        result["time_norm"] = result[time_column].map(normalize_time_string)
    else:
        result["time_norm"] = ""
    result["title_key_current"] = result["title_current"].fillna("")
    result["title_key_relaxed"] = result["title_relaxed"].fillna("")
    result["full_key_current"] = result["title_key_current"] + "|" + result["date_norm"].fillna("") + "|" + result["time_norm"].fillna("")
    result["full_key_relaxed"] = result["title_key_relaxed"] + "|" + result["date_norm"].fillna("") + "|" + result["time_norm"].fillna("")
    return result


def add_person_normalizations(frame: pd.DataFrame, name_column: str, description_column: str | None = None) -> pd.DataFrame:
    result = frame.copy()
    if name_column in result.columns:
        result["name_current"] = result[name_column].map(normalize_name)
        result["name_relaxed"] = result[name_column].map(normalize_name_relaxed)
    else:
        result["name_current"] = ""
        result["name_relaxed"] = ""
    if description_column and description_column in result.columns:
        result["description_norm"] = result[description_column].map(normalize_text)
    else:
        result["description_norm"] = ""
    return result


candidate_episode_norms = add_episode_normalizations(candidate_episodes, "sendungstitel", "publication_data_0", "publication_time_0")

zdf_episode_rows = episodes_df.copy()
if "deterministic_alignment_method" in zdf_episode_rows.columns:
    zdf_episode_rows = zdf_episode_rows[zdf_episode_rows["deterministic_alignment_method"].astype(str).eq("seed_zdf_episode")].copy()
else:
    zdf_episode_rows = zdf_episode_rows.iloc[0:0].copy()

wikidata_episode_rows = episodes_df.copy()
if "deterministic_alignment_method" in wikidata_episode_rows.columns:
    wikidata_episode_rows = wikidata_episode_rows[wikidata_episode_rows["deterministic_alignment_method"].astype(str).eq("seed_wikidata_episode")].copy()
else:
    wikidata_episode_rows = wikidata_episode_rows.iloc[0:0].copy()

aligned_episode_rows = episodes_df.copy()
if "deterministic_alignment_method" in aligned_episode_rows.columns:
    aligned_episode_rows = aligned_episode_rows[
        aligned_episode_rows["deterministic_alignment_method"].astype(str).str.startswith("time_program_match_")
    ].copy()
else:
    aligned_episode_rows = aligned_episode_rows.iloc[0:0].copy()

zdf_episode_norms = add_episode_normalizations(zdf_episode_rows, "broadcasting_program_key", "source_zdf_value", "publication_time")
wikidata_episode_norms = add_episode_normalizations(wikidata_episode_rows, "source_wikidata_value")
aligned_episode_norms = add_episode_normalizations(aligned_episode_rows, "source_zdf_value", "publication_date", "publication_time")

mention_person_norms = add_person_normalizations(mention_persons, "name", "description")
person_projection_norms = add_person_normalizations(persons_df, "source_zdf_value", "source_fernsehserien_value")

candidate_episode_norms["current_title_match"] = candidate_episode_norms["title_key_current"].isin(set(zdf_episode_norms["title_key_current"]))
candidate_episode_norms["relaxed_title_match"] = candidate_episode_norms["title_key_relaxed"].isin(set(zdf_episode_norms["title_key_relaxed"]))
candidate_episode_norms["full_match"] = candidate_episode_norms["full_key_current"].isin(set(zdf_episode_norms["full_key_current"]))

candidate_episode_norms["time_program_aligned"] = candidate_episode_norms["title_key_current"].isin(set(aligned_episode_norms["title_key_current"]))

wikidata_episode_norms["current_title_match"] = wikidata_episode_norms["title_key_current"].isin(set(candidate_episode_norms["title_key_current"]))
wikidata_episode_norms["relaxed_title_match"] = wikidata_episode_norms["title_key_relaxed"].isin(set(candidate_episode_norms["title_key_relaxed"]))

mention_person_norms["current_name_match"] = mention_person_norms["name_current"].isin(set(person_projection_norms["name_current"]))
mention_person_norms["relaxed_name_match"] = mention_person_norms["name_relaxed"].isin(set(person_projection_norms["name_relaxed"]))
person_projection_norms["current_name_match"] = person_projection_norms["name_current"].isin(set(mention_person_norms["name_current"]))
person_projection_norms["relaxed_name_match"] = person_projection_norms["name_relaxed"].isin(set(mention_person_norms["name_relaxed"]))

episode_stats = pd.DataFrame(
    [
        {
            "comparison": "candidate -> zdf seed",
            "title_exact_current": int(candidate_episode_norms["current_title_match"].sum()),
            "title_exact_relaxed": int(candidate_episode_norms["relaxed_title_match"].sum()),
            "full_key_exact": int(candidate_episode_norms["full_match"].sum()),
            "time_program_aligned": int(candidate_episode_norms["time_program_aligned"].sum()),
            "candidate_rows": len(candidate_episode_norms),
            "projection_rows": len(zdf_episode_norms),
        },
        {
            "comparison": "candidate -> wikidata seed",
            "title_exact_current": int(wikidata_episode_norms["current_title_match"].sum()),
            "title_exact_relaxed": int(wikidata_episode_norms["relaxed_title_match"].sum()),
            "full_key_exact": int(wikidata_episode_norms["full_key_current"].isin(set(candidate_episode_norms["full_key_current"])) .sum()),
            "time_program_aligned": len(aligned_episode_norms),
            "candidate_rows": len(candidate_episode_norms),
            "projection_rows": len(wikidata_episode_norms),
        },
    ]
)
episode_stats["title_current_coverage"] = episode_stats["title_exact_current"] / episode_stats["candidate_rows"].where(episode_stats["candidate_rows"] > 0, 1)
episode_stats["title_relaxed_coverage"] = episode_stats["title_exact_relaxed"] / episode_stats["candidate_rows"].where(episode_stats["candidate_rows"] > 0, 1)
episode_stats["full_key_coverage"] = episode_stats["full_key_exact"] / episode_stats["candidate_rows"].where(episode_stats["candidate_rows"] > 0, 1)
episode_stats["time_program_alignment_coverage"] = episode_stats["time_program_aligned"] / episode_stats["candidate_rows"].where(episode_stats["candidate_rows"] > 0, 1)

person_stats = pd.DataFrame(
    [
        {
            "comparison": "mention_persons -> projection seeds",
            "title_exact_current": int(mention_person_norms["current_name_match"].sum()),
            "title_exact_relaxed": int(mention_person_norms["relaxed_name_match"].sum()),
            "candidate_rows": len(mention_person_norms),
            "projection_rows": len(person_projection_norms),
        }
    ]
)
person_stats["title_current_coverage"] = person_stats["title_exact_current"] / person_stats["candidate_rows"].where(person_stats["candidate_rows"] > 0, 1)
person_stats["title_relaxed_coverage"] = person_stats["title_exact_relaxed"] / person_stats["candidate_rows"].where(person_stats["candidate_rows"] > 0, 1)

print("Episode match stats")
print(episode_stats.to_string(index=False))
print()
print("Aligned episode method counts")
if "deterministic_alignment_method" in aligned_episode_rows.columns:
    print(aligned_episode_rows["deterministic_alignment_method"].astype(str).value_counts(dropna=False).to_string())
else:
    print("No aligned episode methods present")
print()
print("Person match stats")
print(person_stats.to_string(index=False))

changed_episode_rows = candidate_episode_norms[candidate_episode_norms["title_current"] != candidate_episode_norms["title_relaxed"]]
changed_person_rows = mention_person_norms[mention_person_norms["name_current"] != mention_person_norms["name_relaxed"]]

print()
print("Episode rows changed by relaxed normalization")
print(changed_episode_rows[["sendungstitel", "title_current", "title_relaxed"]].head(15).to_string(index=False))
print()
print("Person rows changed by relaxed normalization")
print(changed_person_rows[["name", "name_current", "name_relaxed"]].head(15).to_string(index=False))

## 4. Identify Exact and Near Matches We Are Still Missing

In [ ]:
def near_miss_table(source_frame: pd.DataFrame, source_column: str, target_frame: pd.DataFrame, target_column: str, *, normalizer=normalize_text, limit: int = 15, max_source_rows: int = 250) -> pd.DataFrame:
    if source_frame.empty or target_frame.empty or source_column not in source_frame.columns or target_column not in target_frame.columns:
        return pd.DataFrame()
    target_values = target_frame[target_column].dropna().astype(str)
    rows = []
    for _, source_row in source_frame.head(max_source_rows).iterrows():
        source_value = source_row.get(source_column, "")
        if not str(source_value).strip():
            continue
        best_index, best_score, best_value = best_match_index(source_value, target_values, normalizer)
        rows.append(
            {
                source_column: source_value,
                "best_match": best_value,
                "score": round(best_score, 4),
                "best_match_index": best_index,
            }
        )
    if not rows:
        return pd.DataFrame()
    return pd.DataFrame(rows).sort_values(["score", source_column], ascending=[False, True]).head(limit)


episode_candidates_no_zdf_match = candidate_episode_norms[~candidate_episode_norms["current_title_match"]].copy()
episode_candidates_no_wikidata_match = candidate_episode_norms[~candidate_episode_norms["title_key_current"].isin(set(wikidata_episode_norms["title_key_current"]))].copy()
episode_candidates_not_time_aligned = candidate_episode_norms[~candidate_episode_norms["time_program_aligned"]].copy()

person_candidates_no_current_match = mention_person_norms[~mention_person_norms["current_name_match"]].copy()
person_candidates_no_relaxed_match = mention_person_norms[~mention_person_norms["relaxed_name_match"]].copy()

episode_near_misses_zdf = near_miss_table(
    episode_candidates_no_zdf_match,
    "sendungstitel",
    zdf_episode_rows,
    "broadcasting_program_key",
    normalizer=normalize_episode_title,
)
episode_near_misses_wikidata = near_miss_table(
    episode_candidates_no_wikidata_match,
    "sendungstitel",
    wikidata_episode_rows,
    "source_wikidata_value",
    normalizer=normalize_episode_title_relaxed,
)
episode_near_misses_time_aligned = near_miss_table(
    episode_candidates_not_time_aligned,
    "sendungstitel",
    aligned_episode_rows,
    "source_zdf_value",
    normalizer=normalize_episode_title,
)

person_near_misses_zdf = near_miss_table(
    person_candidates_no_current_match,
    "name",
    persons_df,
    "source_zdf_value",
    normalizer=normalize_name,
)
person_near_misses_fernsehserien = near_miss_table(
    person_candidates_no_current_match,
    "name",
    persons_df,
    "source_fernsehserien_value",
    normalizer=normalize_name_relaxed,
)

episode_near_misses = pd.concat(
    [
        episode_near_misses_zdf.assign(reference="zdf"),
        episode_near_misses_wikidata.assign(reference="wikidata"),
        episode_near_misses_time_aligned.assign(reference="time_program_aligned"),
    ],
    ignore_index=True,
)
person_near_misses = pd.concat(
    [
        person_near_misses_zdf.assign(reference="zdf"),
        person_near_misses_fernsehserien.assign(reference="fernsehserien"),
    ],
    ignore_index=True,
)

print("Episode near misses")
if not episode_near_misses.empty:
    print(episode_near_misses.head(20).to_string(index=False))
else:
    print("No episode near misses found")

print()
print("Person near misses")
if not person_near_misses.empty:
    print(person_near_misses.head(20).to_string(index=False))
else:
    print("No person near misses found")

print()
print("Episodes not covered by time_program_aligned matcher")
print(episode_candidates_not_time_aligned[["sendungstitel", "publication_data_0", "publication_time_0"]].head(20).to_string(index=False))

print()
print("Episode rows that become exact matches after relaxed normalization")
print(candidate_episode_norms[candidate_episode_norms["title_current"] != candidate_episode_norms["title_relaxed"]][["sendungstitel", "title_current", "title_relaxed"]].head(20).to_string(index=False))
print()
print("Person rows that become exact matches after relaxed normalization")
print(mention_person_norms[mention_person_norms["name_current"] != mention_person_norms["name_relaxed"]][["name", "name_current", "name_relaxed"]].head(20).to_string(index=False))

## 5. Coverage Report and Fix Checklist

In [ ]:
def classify_episode_gap(value: object) -> str:
    text = "" if pd.isna(value) else str(value)
    reasons = []
    if re.search(r"\([^)]*\)", text):
        reasons.append("parenthetical suffix")
    if re.search(r"\b\d{1,2}[./]\d{1,2}[./]\d{2,4}\b", text):
        reasons.append("date token")
    if text != text.strip():
        reasons.append("outer whitespace")
    if re.match(r"^[\-–—•]+", text):
        reasons.append("leading punctuation")
    if re.search(r"[\u00c0-\u017f]", text):
        reasons.append("accented characters")
    if re.search(r"\b(vom|von|der|die|das)\b", text, flags=re.IGNORECASE):
        reasons.append("function words")
    return "; ".join(reasons) if reasons else "other"


def classify_person_gap(value: object) -> str:
    text = "" if pd.isna(value) else str(value)
    reasons = []
    if re.search(r"\([^)]*\)", text):
        reasons.append("parenthetical descriptor")
    if text != text.strip():
        reasons.append("outer whitespace")
    if re.match(r"^[\-–—•]+", text):
        reasons.append("leading punctuation")
    if re.search(r"\b(Dr\.?|Prof\.?|Dipl\.?|Ing\.?|Diplom)\b", text, flags=re.IGNORECASE):
        reasons.append("honorific")
    if re.search(r"[\"'«»]", text):
        reasons.append("quotes")
    if re.search(r"\b(aka|alias|genannt)\b", text, flags=re.IGNORECASE):
        reasons.append("alias marker")
    return "; ".join(reasons) if reasons else "other"


coverage_report = pd.concat(
    [
        episode_stats.assign(kind="episodes"),
        person_stats.assign(kind="persons"),
    ],
    ignore_index=True,
    sort=False,
)
print(coverage_report.to_string(index=False))

print()
print("Episode gap reasons")
print(candidate_episode_norms.assign(gap_reason=candidate_episode_norms["sendungstitel"].map(classify_episode_gap))["gap_reason"].value_counts().to_string())

print()
print("Person gap reasons")
print(mention_person_norms.assign(gap_reason=mention_person_norms["name"].map(classify_person_gap))["gap_reason"].value_counts().to_string())

print()
print("Most actionable episode candidates")
print(episode_near_misses.head(10).to_string(index=False))
print()
print("Most actionable person candidates")
print(person_near_misses.head(10).to_string(index=False))

print()
print("Proposed fix checklist")
print("1. Keep Layer-2 deterministic matching driven by publication date + normalized program label; preserve orphan policy on ambiguous keys.")
print("2. Add stricter time handling (seconds normalization and optional tolerance windows) to raise full-key coverage without forced matches.")
print("3. Extend Wikidata date extraction beyond '(dd. Monat yyyy)' labels when equivalent high-confidence signals exist.")
print("4. Re-run production notebook, then re-run this diagnostics notebook and compare time_program_alignment_coverage.")

## 6. Re-run After Applying Fixes

After promoting the normalization rules that look safe here, rerun `31_entity_disambiguation.ipynb` and compare the episode and person coverage tables again. The goal is to keep tightening the obvious-match gap without changing the deterministic ordering or the event-sourced replay behavior.